In [1]:
import pandas as pd

In [23]:
task2 = "https://github.com/PleschNA/testGA/raw/main/task2.xlsx"
df = pd.read_excel(task2)
df["Дата"] = pd.to_datetime(df["Дата"])
print(df.head())
print(df["Код товара"].nunique(), "уникальных товаров")


        Дата  Код товара  Количество продано  Розничная цена  \
0 2024-06-01  3226300004                   9             271   
1 2024-06-02  3226300004                  15             271   
2 2024-06-03  3226300004                  12             271   
3 2024-06-04  3226300004                  17             271   
4 2024-06-05  3226300004                  13             271   

   Цена с учётом скидки  
0                 271.0  
1                 271.0  
2                 216.8  
3                 216.8  
4                 216.8  
10 уникальных товаров


In [10]:
#Есть ли скидка 
df["Скидка_есть"] = df["Цена с учётом скидки"] < df["Розничная цена"]
df["Скидка_%"] = (1 - df["Цена с учётом скидки"] / df["Розничная цена"]) * 100

In [11]:
results = []
for code, g in df.groupby("Код товара"):
    base = g[~g["Скидка_есть"]]["Количество продано"].mean()
    disc = g[g["Скидка_есть"]]
    if disc.empty or pd.isna(base):
        continue
    disc_sales = disc["Количество продано"].mean()
    disc_pct = disc["Скидка_%"].mean()
    
    # Эластичность: (% роста продаж) / (% снижения цены)
    growth = (disc_sales - base) / base * 100
    elasticity = growth / disc_pct if disc_pct > 0 else None
    
    # Нужная скидка для x2
    needed_discount = 100 / elasticity if elasticity and elasticity > 0 else None
    
    results.append({
        "Код товара": code,
        "Продажи без скидки": round(base, 1),
        "Продажи со скидкой": round(disc_sales, 1),
        "Рост, %": round(growth, 1),
        "Скидка, %": round(disc_pct, 1),
        "Эластичность": round(elasticity, 2) if elasticity else None,
        "Скидка для ×2, %": round(needed_discount, 1) if needed_discount and needed_discount < 90 else "Недостижимо",
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

 Код товара  Продажи без скидки  Продажи со скидкой  Рост, %  Скидка, %  Эластичность  Скидка для ×2, %
 3226300004                10.2                24.5    139.6       28.3          4.93              20.3
 3234800007                10.3                22.4    117.3       22.5          5.21              19.2
14011400047                10.1                28.7    185.6       32.5          5.71              17.5
14012900005                 9.6                27.1    183.3       28.3          6.47              15.5
15110900016                10.2                18.7     82.8       21.7          3.82              26.2
15111300006                 9.4                23.8    152.6       24.0          6.36              15.7
15111300017                 5.0                10.4    107.5       17.5          6.14              16.3
19000000363                10.5                22.2    112.3       22.5          4.99              20.0
19000013047                10.2                29.6    190.8    

In [28]:
# Сводка
print("Средняя эластичность:", round(res_df["Эластичность"].mean(), 2))
print("Средняя скидка для ×2:", round(res_df["Скидка для ×2, %"].apply(
    lambda x: x if isinstance(x, (int, float)) else None).dropna().mean(), 1), "%")
print()
print("Все товары эластичны?", (res_df["Эластичность"] > 1).all())

Средняя эластичность: 5.37
Средняя скидка для ×2: 19.2 %

Все товары эластичны? True
